# Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

import os
from tqdm import tqdm

from CustomLayers import (
    DummyLinear, 
    QuantizedLinearGlobalTorch, 
    GlobalQuantLinearTriton
)
from benchmark.perplexity import measure_ppl

d:\anaconda3\envs\torch_env_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Utils

In [2]:
def change_linear_layer(model, new_layer, device):
    for layer in model.model.layers:
        layer.self_attn.q_proj = new_layer(layer.self_attn.q_proj, device)
        layer.self_attn.k_proj = new_layer(layer.self_attn.k_proj, device)
        layer.self_attn.v_proj = new_layer(layer.self_attn.v_proj, device)
        layer.self_attn.o_proj = new_layer(layer.self_attn.o_proj, device)

        layer.mlp.gate_proj = new_layer(layer.mlp.gate_proj, device)
        layer.mlp.up_proj = new_layer(layer.mlp.up_proj, device)
        layer.mlp.down_proj = new_layer(layer.mlp.down_proj, device)

    model.lm_head = new_layer(model.lm_head, device)
    torch.cuda.empty_cache()

    return model

def calc_model_size(model):
    param_mem = 0.
    buffer_mem = 0.
    for param in model.parameters():
        param_mem += param.nelement() * param.element_size()
    for buffer in model.buffers():
        buffer_mem += buffer.nelement() * buffer.element_size()

    return (param_mem + buffer_mem) / (2 ** 30)


def time_inference(model, tokenizer, device, text, max_length=50):

    inputs = tokenizer(text, return_tensors='pt').to(device)
    
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length, 
                               pad_token_id=tokenizer.eos_token_id)
    end_time = time.time()
    
    inference_time = end_time - start_time
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return inference_time, generated_text


def verify_layer_accuracy(orig_model, quant_model, tokenizer, text='hello'):
    inputs = tokenizer(text, return_tensors='pt').to(orig_model.device)
    inputs_ = tokenizer(text, return_tensors='pt').to(quant_model.device)

    with torch.no_grad():
        orig_outputs = orig_model(**inputs, output_hidden_states=True)

    with torch.no_grad():
        quant_outputs = quant_model(**inputs_, output_hidden_states=True)

    for i, (orig_hidden, quant_hidden) in enumerate(zip(orig_outputs.hidden_states, quant_outputs.hidden_states)):
        print(orig_hidden.shape, quant_hidden.shape)
        mse = F.mse_loss(orig_hidden.cpu(), quant_hidden.cpu())
        cos_sim = F.cosine_similarity(orig_hidden.flatten().cpu(), quant_hidden.flatten().cpu(), dim=0)
        print(f'layer {i}: mse = {mse:.6f}, cosine_sim = {cos_sim:.6f}')

# Load Data

In [3]:
# raw_datasets = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test") # longer sequnces
raw_datasets = load_dataset('zhengxuanzenwu/wikitext-2-split-128', split='test')

Repo card metadata block was not found. Setting CardData to empty.


In [4]:
prompts = [x['text'] for x in raw_datasets if len(x['text']) > 0]
print('Number of sequences:', len(prompts))

Number of sequences: 8192


# Load Model

## Baseline

In [5]:
model_id = 'unsloth/Llama-3.2-1B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='cuda:0')

## Modified

In [ ]:
custom_model = AutoModelForCausalLM.from_pretrained(model_id, device_map='cuda:0')
custom_model = change_linear_layer(custom_model, GlobalQuantLinearTriton, custom_model.device)

# Perplexity benchmark

In [9]:
orig_ppl, orig_time = measure_ppl(prompts, model, tokenizer)

100%|██████████| 8192/8192 [07:43<00:00, 17.69it/s]


Perplexity: 345.0570
Mean time per sample: 0.057 s


In [10]:
custom_ppl, custom_time = measure_ppl(prompts, custom_model, tokenizer)

100%|██████████| 8192/8192 [08:02<00:00, 16.98it/s]



Perplexity: 345.0570
Mean time per sample: 0.059 s


# Quant tests

In [ ]:
x = nn.Linear(1000, 1000, bias=True, dtype=torch.float32).to('cuda:1')
y = torch.randn(1000, 1000, dtype=torch.float32, requires_grad=False).to('cuda:1')
out_real = x(y)
l = QuantizedLinearGlobalTorch(x, 'cuda:1')
out_quant = l(y)

print(F.l1_loss(out_real, out_quant))

del x
del y
del l
torch.cuda.empty_cache()

In [ ]:
custom_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, device_map='cuda:1')
custom_model = change_linear_layer(custom_model, QuantizedLinearGlobalTorch, custom_model.device)

In [ ]:
custom_model

In [ ]:
text = 'Hello how is the weather?'
orig_model_size = calc_model_size(model)
orig_inf_time, orig_inf_text = time_inference(model, tokenizer, model.device, text)
print('%f GB; %f s.;\n%s' % (orig_model_size, orig_inf_time, orig_inf_text))

In [ ]:
text = 'Hello how is the weather?'
quant_model_size = calc_model_size(custom_model)
quant_inf_time, quant_inf_text = time_inference(custom_model, tokenizer, custom_model.device, text)
print('%f GB; %f s.;\n%s' % (quant_model_size, quant_inf_time, quant_inf_text))

In [ ]:
verify_layer_accuracy(model, custom_model, tokenizer)